In [1]:
# Cài đặt ultralytics nếu chưa có
# !pip install ultralytics

In [1]:
import os
import torch
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [2]:
# Config
DATA_YAML   = "../data/yolo_dataset/data.yaml"
MODEL_SIZE  = "yolov8s.pt" 
EPOCHS      = 50
IMG_SIZE    = 640
BATCH_SIZE  = 16
PROJECT_DIR = "../models/yolo"
RUN_NAME    = "rice_leaf_detect"

print(f"Device : {'GPU ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Data   : {os.path.abspath(DATA_YAML)}")

Device : CPU
Data   : D:\Python Projects\RicAI_CV_main\ric_cv\data\yolo_dataset\data.yaml


In [3]:
# 1. Load pretrained YOLOv8n
model = YOLO(MODEL_SIZE)
print(f" Loaded {MODEL_SIZE}")

 Loaded yolov8s.pt


In [4]:
# FIX ID .TXT
BASE_LABEL_DIR = "../data/yolo_dataset/labels"

for split in ["train", "val", "test"]:
    label_dir = os.path.join(BASE_LABEL_DIR, split)
    
    if not os.path.exists(label_dir):
        continue
    
    for file in os.listdir(label_dir):
        path = os.path.join(label_dir, file)
        
        with open(path, "r") as f:
            lines = f.readlines()
        
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                parts[0] = "0"  # fix class id
                new_lines.append(" ".join(parts))
        
        with open(path, "w") as f:
            f.write("\n".join(new_lines))

print(" Fixed ALL labels (train/val/test)")

 Fixed ALL labels (train/val/test)


In [5]:
# 2. Train
results = model.train(
    data      = DATA_YAML,
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH_SIZE,
    project   = PROJECT_DIR,
    name      = RUN_NAME,
    exist_ok  = True,
    patience  = 15,          # Early stopping
    save      = True,
    plots     = True,
    
    # Augmentation (nhẹ - ảnh đã augment)
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    degrees   = 10.0,
    translate = 0.1,
    scale     = 0.5,
    fliplr    = 0.5,
    flipud    = 0.2,
    mosaic    = 0.5,
)

print("\n Training complete!")

New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.12.2 torch-2.3.1+cpu CPU (11th Gen Intel Core i5-11400H @ 2.70GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/yolo_dataset/data.yaml, degrees=10.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.5, multi_scale

In [6]:
print(results.save_dir)

D:\Python Projects\RicAI_CV_main\ric_cv\notebooks\runs\models\yolo\rice_leaf_detect


In [9]:
# 3. Đường dẫn model tốt nhất 
best_model_path = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'best.pt')
print(f"Best model: {best_model_path}")
print(f"Exists    : {os.path.exists(best_model_path)}")

Best model: ../models/yolo\rice_leaf_detect\weights\best.pt
Exists    : True


In [10]:
# 4. Validate
best_model = YOLO(best_model_path)
metrics = best_model.val(data=DATA_YAML)

print("\n" + "="*60)
print("VALIDATION METRICS")
print("="*60)
print(f"mAP50         : {metrics.box.map50:.4f}")
print(f"mAP50-95      : {metrics.box.map:.4f}")
print(f"Precision     : {metrics.box.mp:.4f}")
print(f"Recall        : {metrics.box.mr:.4f}")

Ultralytics 8.4.37  Python-3.12.2 torch-2.3.1+cpu CPU (11th Gen Intel Core i5-11400H @ 2.70GHz)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 21.141.5 MB/s, size: 210.0 KB)
val: Scanning D:\Python Projects\RicAI_CV_main\ric_cv\data\yolo_dataset\labels\val.cache... 240 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 240/240  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 3.3s/it 49.3s3.2ss
                   all        240        267      0.892      0.801      0.855      0.594
Speed: 1.5ms preprocess, 191.0ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to D:\Python Projects\RicAI_CV_main\ric_cv\notebooks\runs\detect\val2

VALIDATION METRICS
mAP50         : 0.8552
mAP50-95      : 0.5943
Precision     : 0.8924
Recall        : 0.8015


In [11]:
# 5. Hiển thị biểu đồ training 
results_dir = os.path.join(PROJECT_DIR, RUN_NAME)
plot_files = ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']

for plot_file in plot_files:
    path = os.path.join(results_dir, plot_file)
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(12, 6))
        plt.imshow(img)
        plt.title(plot_file)
        plt.axis('off')
        plt.show()
    else:
        print(f"  {plot_file} not found")

<Figure size 1200x600 with 1 Axes>

<Figure size 1200x600 with 1 Axes>

<Figure size 1200x600 with 1 Axes>

In [12]:
# 6. Export sang ONNX (optional - để dùng cross-platform)
# best_model.export(format='onnx')
# print("Exported to ONNX")

# 7. Copy best.pt ra thư mục models chính
import shutil
os.makedirs("../models", exist_ok=True)
dst = "../models/yolov8s_rice_leaf.pt"
shutil.copy(best_model_path, dst)
print(f" Saved best model -> {dst}")

print("\n READY for 10_apply_yolo_crop_all.ipynb")

 Saved best model -> ../models/yolov8s_rice_leaf.pt

 READY for 10_apply_yolo_crop_all.ipynb
